# Audio Reconstruction Evaluation

**Sezione 1 — Conditioning modes**: `ip_adapter` vs `brain_only` vs `empty_plus`  
Metriche: CLAP similarity, log-mel distance, Mel-SSIM + Wilcoxon

**Sezione 2 — ROI ablation**: per ogni subset ROI  
Metrica: CLAP shift + Wilcoxon one-sample + matrice ROI x semantic class

In [19]:
# !pip install transformers librosa scipy scikit-image pandas matplotlib seaborn --quiet

In [ ]:
import re, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wilcoxon, false_discovery_control
from skimage.metrics import structural_similarity as ssim

warnings.filterwarnings('ignore')
sns.set_context('talk')
sns.set_style('whitegrid')

BASE = Path('.../data')
DIRS = {
    'gt':          BASE / 'decoded_ip2',
    'ip_adapter':  BASE / 'decoded_ip2',
    'brain_only':  BASE / 'decoded_brain_only',
    'empty_plus':  BASE / 'decoded_plus',
    'brain_to_text': BASE / 'output_brain_to_text',
    'roi_subset':  BASE / 'decoded_roi_subset',
}
SOUND_NAMES_PATH = '.../data/SoundNames.npy'
CLAP_MODEL_ID    = 'laion/clap-htsat-unfused'
TARGET_SR        = 48000
N_MELS, N_FFT, HOP_LENGTH = 64, 1024, 256
ALPHA            = 0.05
DEVICE           = 'cuda:3' if torch.cuda.is_available() else 'cpu'

print('Device:', DEVICE)
for k, v in DIRS.items():
    print(f'  {k:15s}: {"OK" if v.exists() else "NOT FOUND"}')

Device: cuda:3
  gt             : OK
  ip_adapter     : OK
  brain_only     : OK
  empty_plus     : OK
  brain_to_text  : OK
  roi_subset     : OK


In [ ]:
from transformers import ClapModel, ClapProcessor
processor_clap = ClapProcessor.from_pretrained(CLAP_MODEL_ID)
model_clap     = ClapModel.from_pretrained(CLAP_MODEL_ID).to(DEVICE).eval()
print(f'CLAP load: {CLAP_MODEL_ID}')

In [ ]:
# ── Utility ──────────────────────────────────────────────────────────────────
def load_audio_48k(path, duration_s=None):
    wav, _ = librosa.load(path, sr=TARGET_SR, mono=True)
    if duration_s is not None:
        n = int(TARGET_SR * duration_s)
        wav = np.pad(wav, (0, max(0, n - len(wav))))[:n]
    return wav.astype(np.float32)

def align(a, b):
    n = min(len(a), len(b)); return a[:n], b[:n]

def compute_log_mel_norm(wav):
    mel = librosa.feature.melspectrogram(y=wav, sr=TARGET_SR,
                                         n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
    lm  = librosa.power_to_db(mel, ref=np.max)
    lo, hi = lm.min(), lm.max()
    return np.zeros_like(lm) if (hi - lo) < 1e-6 else (lm - lo) / (hi - lo)

@torch.no_grad()
def get_clap_embeddings(paths, batch_size=8):
    out = []
    for i in range(0, len(paths), batch_size):
        wavs   = [load_audio_48k(p) for p in paths[i:i+batch_size]]
        inputs = processor_clap(audios=wavs, return_tensors='pt',
                                sampling_rate=TARGET_SR, padding=True)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        emb    = model_clap.get_audio_features(**inputs)
        emb    = emb / emb.norm(dim=-1, keepdim=True).clamp(min=1e-8)
        out.append(emb.cpu().float().numpy())
    return np.vstack(out)

def cosine_sim(a, b): return float(np.dot(a, b))

def parse_sid(fname):
    m = re.search(r'sound(\d+)', fname); return int(m.group(1)) if m else None

def build_gt_index(d):
    idx = {}
    for f in d.glob('gt_*.wav'):
        i = parse_sid(f.name)
        if i is not None and i not in idx: idx[i] = str(f)
    return idx

def build_pred_list(d):
    return [(parse_sid(f.name), str(f)) for f in sorted(d.glob('pred_*.wav'))
            if parse_sid(f.name) is not None]

def safe_ssim(a, b):
    t = min(a.shape[1], b.shape[1]); a, b = a[:,:t], b[:,:t]
    win = max(3, min(7, a.shape[0], t))
    win = win if win % 2 == 1 else win - 1
    return float(ssim(a, b, data_range=1.0, win_size=win)) if a.shape[0]>=3 and t>=3 else float('nan')

def sig_label(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'n.s.'

def fdr_correct(df, p_col='p_raw'):
    df = df.copy()
    valid = df[p_col].notna()
    pv = df.loc[valid, p_col].values
    try:
        pfdr = false_discovery_control(pv, method='bh')
    except Exception:
        from statsmodels.stats.multitest import multipletests
        _, pfdr, _, _ = multipletests(pv, method='fdr_bh')
    df.loc[valid, 'p_fdr'] = pfdr
    df['sig'] = df['p_fdr'].apply(sig_label)
    return df

# ── Carica sound metadata ────────────────────────────────────────────────────
def parse_semantic_class(name):
    m = re.match(r'stim\d+_cat\d+_([a-zA-Z]+)_exemp\d+', str(name))
    if m is None: raise ValueError(f'unexpected format: {name}')
    return m.group(1).lower()

sound_names = np.load(SOUND_NAMES_PATH, allow_pickle=True)
sound_meta  = pd.DataFrame([{'sound_idx': i, 'semantic_class': parse_semantic_class(n)}
                             for i, n in enumerate(sound_names)])
print(f'Sound metadata: {len(sound_meta)} sounds, {sound_meta["semantic_class"].nunique()} classes')
display(sound_meta['semantic_class'].value_counts().sort_index().rename('n').to_frame())

gt_index = build_gt_index(DIRS['gt'])
print(f'Ground truth: {len(gt_index)} unique sound_id found')

---
# Section 1 — Compare conditioning modes

In [ ]:
def evaluate_conditioning(pred_dir, gt_idx, label):
    matched    = [(i, p) for i, p in build_pred_list(pred_dir) if i in gt_idx]
    if not matched: print(f'[WARN] not found in {pred_dir}'); return pd.DataFrame()
    print(f'  {label}: {len(matched)} matched pairs')
    pred_paths = [p for _, p in matched]
    gt_paths   = [gt_idx[i] for i, _ in matched]
    sound_idxs = [i for i, _ in matched]
    print('    → CLAP pred ...'); emb_pred = get_clap_embeddings(pred_paths)
    print('    → CLAP gt ...');   emb_gt   = get_clap_embeddings(gt_paths)
    rows = []
    for i, idx in enumerate(sound_idxs):
        wav_p = load_audio_48k(pred_paths[i])
        wav_g = load_audio_48k(gt_paths[i], duration_s=len(wav_p)/TARGET_SR)
        wav_p, wav_g = align(wav_p, wav_g)
        mel_p, mel_g = compute_log_mel_norm(wav_p), compute_log_mel_norm(wav_g)
        rows.append({'condition': label, 'sound_idx': idx,
                     'clap_similarity': cosine_sim(emb_pred[i], emb_gt[i]),
                     'log_mel_dist': float(np.mean(np.abs(mel_p - mel_g))),
                     'mel_ssim': safe_ssim(mel_p, mel_g)})
    return pd.DataFrame(rows)

results_list = []
for label, pred_dir in [('ip_adapter', DIRS['ip_adapter']),
                         ('brain_only', DIRS['brain_only']),
                         ('empty_plus', DIRS['empty_plus']),
                         ('brain_to_text', DIRS['brain_to_text'])]:
    print(f'\n→ {label}')
    df = evaluate_conditioning(pred_dir, gt_index, label)
    if not df.empty: results_list.append(df)

df_cond = pd.concat(results_list, ignore_index=True)
print('\n=== Mean per conditioning mode ===')
display(df_cond.groupby('condition')[['clap_similarity','log_mel_dist','mel_ssim']].mean().round(4))

In [ ]:
COLORS = {'ip_adapter':'#1D9E75','brain_only':'#D85A30','empty_plus':'#378ADD','brain_to_text':'#9B59B6'}
ORDER  = ['ip_adapter','brain_only','empty_plus','brain_to_text']
fig, axes = plt.subplots(1, 3, figsize=(24, 6))
for ax, (metric, title) in zip(axes, {'clap_similarity':'CLAP similarity ↑',
                                        'log_mel_dist':'Log-mel distance ↓',
                                        'mel_ssim':'Mel-SSIM ↑'}.items()):
    sns.boxplot(data=df_cond, x='condition', y=metric, palette=COLORS, order=ORDER, ax=ax,
                flierprops=dict(marker='o', markersize=3, alpha=0.4))
    sns.stripplot(data=df_cond, x='condition', y=metric, palette=COLORS, order=ORDER, ax=ax,
                  size=3, alpha=0.3, jitter=True)
    ax.set_title(title, fontsize=12); ax.set_xlabel(''); ax.tick_params(axis='x', rotation=15)
fig.suptitle('compare conditioning modes — test set', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

In [6]:
ip_df = df_cond.loc[df_cond['condition']=='ip_adapter', ['sound_idx','clap_similarity','mel_ssim']]
summary_rows = [{'condition':'ip_adapter',
                 'mean_clap': ip_df['clap_similarity'].mean(), 'mean_ssim': ip_df['mel_ssim'].mean(),
                 'delta_clap':0.0,'delta_ssim':0.0,'wilcoxon_p_clap':float('nan'),'wilcoxon_p_ssim':float('nan')}]
for cond in ['brain_only','empty_plus','brain_to_text']:
    other  = df_cond.loc[df_cond['condition']==cond, ['sound_idx','clap_similarity','mel_ssim']]
    merged = ip_df.merge(other, on='sound_idx', suffixes=('_ip','_other'))
    def wp(c1,c2): return wilcoxon(merged[c1],merged[c2]).pvalue if len(merged)>=5 else float('nan')
    summary_rows.append({'condition':cond,
        'mean_clap':other['clap_similarity'].mean(),'mean_ssim':other['mel_ssim'].mean(),
        'delta_clap':other['clap_similarity'].mean()-ip_df['clap_similarity'].mean(),
        'delta_ssim':other['mel_ssim'].mean()-ip_df['mel_ssim'].mean(),
        'wilcoxon_p_clap':wp('clap_similarity_ip','clap_similarity_other'),
        'wilcoxon_p_ssim':wp('mel_ssim_ip','mel_ssim_other')})
display(pd.DataFrame(summary_rows).style
    .format({'mean_clap':'{:.4f}','mean_ssim':'{:.4f}','delta_clap':'{:+.4f}',
             'delta_ssim':'{:+.4f}','wilcoxon_p_clap':'{:.4f}','wilcoxon_p_ssim':'{:.4f}'})
    .background_gradient(subset=['mean_clap','mean_ssim'], cmap='Greens'))

,condition,mean_clap,mean_ssim,delta_clap,delta_ssim,wilcoxon_p_clap,wilcoxon_p_ssim
0,ip_adapter,0.6216,0.3056,+0.0000,+0.0000,nan,nan
1,brain_only,0.4361,0.2726,-0.1855,-0.0330,0.0000,0.0053
2,empty_plus,0.4030,0.2264,-0.2186,-0.0792,0.0000,0.0000
3,brain_to_text,0.3591,0.1573,-0.2624,-0.1482,0.0000,0.0000


# ROI Ablation results

In [86]:
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import torch

from IPython.display import Audio, display
from scipy.stats import wilcoxon
from skimage.metrics import structural_similarity as ssim

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'figure.titlesize': 14,
})


In [ ]:
BASE = Path('.../data')
DIRS = {
    'gt': BASE / 'decoded_ip2',
    'full_model': BASE / 'decoded_ip_mix',
    'roi_subset': BASE / 'decoded_roi_subset',
}
SOUND_NAMES_PATH = '.../data/SoundNames.npy'
CLAP_MODEL_ID = 'laion/clap-htsat-unfused'

TARGET_SR = 48000
N_MELS = 64
N_FFT = 1024
HOP_LENGTH = 256
FMAX = None
DEVICE = 'cuda:6' if torch.cuda.is_available() else 'cpu'

CLASS_ORDER = ['speech', 'voice', 'music', 'animal', 'nature', 'tools']

COND_LABELS = {
    'ground_truth':  'Ground Truth',
    'full_model':    'Full Model',
    'only_HG2hem':   'only HG',
    'only_PT2hem':   'only PT',
    'only_PP2hem':   'only PP',
    'only_mSTG2hem': 'only mSTG',
    'only_pSTG2hem': 'only pSTG',
    'only_aSTG2hem': 'only aSTG',
}

def pretty_cond(cond):
    if cond in COND_LABELS:
        return COND_LABELS[cond]
    if cond.startswith('only_'):
        return cond.replace('only_', 'only ').replace('2hem', '')
    return cond

ONLY_DIRS = sorted([
    d.name for d in DIRS['roi_subset'].iterdir()
    if d.is_dir() and d.name.startswith('only_')
]) if DIRS['roi_subset'].exists() else []

print('Device:', DEVICE)
print('Ablation found:', ONLY_DIRS)
for k, v in DIRS.items():
    print(f'{k:12s}:', 'OK' if v.exists() else 'NOT FOUND', v)


In [ ]:
sound_names = np.load(SOUND_NAMES_PATH, allow_pickle=True)

def parse_semantic_class(name):
    match = re.match(r'stim\d+_cat\d+_([a-zA-Z]+)_exemp\d+', str(name))
    if match is None:
        simple = re.search(r'_cat\d+_([a-zA-Z]+)_', str(name))
        return simple.group(1).lower() if simple else 'unknown'
    return match.group(1).lower()

sound_meta = pd.DataFrame([
    {'sound_id': i, 'sound_class': parse_semantic_class(name)}
    for i, name in enumerate(sound_names)
])

def parse_key(fname):
    m = re.search(r'sound(\d+)(?:_sample(\d+))?', fname)
    if not m:
        return None
    sound_id = int(m.group(1))
    sample_id = int(m.group(2)) if m.group(2) is not None else 0
    return (sound_id, sample_id)

def build_index(folder, prefix):
    out = {}
    for path in sorted(Path(folder).glob(f'{prefix}*.wav')):
        key = parse_key(path.name)
        if key is not None and key not in out:
            out[key] = str(path)
    return out

gt_index = build_index(DIRS['gt'], 'gt_')
full_index = build_index(DIRS['full_model'], 'pred_')
abl_indexes = {cond: build_index(DIRS['roi_subset'] / cond, 'pred_') for cond in ONLY_DIRS}

common_full_keys = sorted(set(gt_index) & set(full_index))
print('GT unique sound_ids:', len(gt_index))
print('Full model matched:', len(common_full_keys))
for cond in ONLY_DIRS:
    n = len(set(gt_index) & set(abl_indexes[cond]))
    print(f'{cond:>18}: {n}')

display(sound_meta['sound_class'].value_counts().reindex(CLASS_ORDER).fillna(0).astype(int).to_frame('n_samples'))


In [89]:
def load_audio_48k(path, duration_s=None):
    wav, _ = librosa.load(path, sr=TARGET_SR, mono=True)
    if duration_s is not None:
        n = int(TARGET_SR * duration_s)
        wav = np.pad(wav, (0, max(0, n - len(wav))))[:n]
    return wav.astype(np.float32)

def align_wav(a, b):
    n = min(len(a), len(b))
    return a[:n], b[:n]

def align_spec(a, b):
    t = min(a.shape[1], b.shape[1])
    return a[:, :t], b[:, :t]

def compute_log_mel_norm(wav):
    mel = librosa.feature.melspectrogram(y=wav, sr=TARGET_SR,
                                         n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
    lm  = librosa.power_to_db(mel, ref=np.max)
    lo, hi = lm.min(), lm.max()
    return np.zeros_like(lm) if (hi - lo) < 1e-6 else (lm - lo) / (hi - lo)

def safe_ssim(a, b):
    t = min(a.shape[1], b.shape[1]); a, b = a[:,:t], b[:,:t]
    win = max(3, min(7, a.shape[0], t))
    win = win if win % 2 == 1 else win - 1
    return float(ssim(a, b, data_range=1.0, win_size=win)) if a.shape[0]>=3 and t>=3 else float('nan')

def safe_corr(a, b):
    a = np.asarray(a).ravel().astype(np.float64)
    b = np.asarray(b).ravel().astype(np.float64)
    if a.size == 0 or b.size == 0:
        return float('nan')
    if np.std(a) < 1e-8 or np.std(b) < 1e-8:
        return float('nan')
    return float(np.corrcoef(a, b)[0, 1])

def cosine_sim(a, b):
    return float(np.dot(a, b))

MEL_FREQS = librosa.mel_frequencies(n_mels=N_MELS, fmin=0, fmax=TARGET_SR / 2 if FMAX is None else FMAX)


In [ ]:
from transformers import ClapModel, ClapProcessor

processor_clap = ClapProcessor.from_pretrained(CLAP_MODEL_ID)
model_clap = ClapModel.from_pretrained(CLAP_MODEL_ID).to(DEVICE).eval()
print(f'CLAP loaded: {CLAP_MODEL_ID}')

@torch.no_grad()
def get_clap_embeddings(paths, batch_size=8):
    out = []
    for i in range(0, len(paths), batch_size):
        wavs = [load_audio_48k(p) for p in paths[i:i+batch_size]]
        inputs = processor_clap(audios=wavs, return_tensors='pt', sampling_rate=TARGET_SR, padding=True)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        emb = model_clap.get_audio_features(**inputs)
        emb = emb / emb.norm(dim=-1, keepdim=True).clamp(min=1e-8)
        out.append(emb.detach().cpu().float().numpy())
    return np.vstack(out)


CLAP caricato: laion/clap-htsat-unfused


In [ ]:
def evaluate_condition(index_pred, label):
    matched = [(key, path) for key, path in sorted(index_pred.items()) if key in gt_index]
    if not matched:
        print(f'[WARN] no matches in {label}')
        return pd.DataFrame()

    print(f'{label}: {len(matched)} matched pairs')
    pred_paths = [path for _, path in matched]
    gt_paths = [gt_index[key] for key, _ in matched]
    keys = [key for key, _ in matched]

    print('  -> CLAP pred ...')
    emb_pred = get_clap_embeddings(pred_paths)
    print('  -> CLAP gt ...')
    emb_gt = get_clap_embeddings(gt_paths)

    rows = []
    for i, key in enumerate(keys):
        sound_id, sample_id = key
        wav_p = load_audio_48k(pred_paths[i])
        wav_g = load_audio_48k(gt_paths[i], duration_s=len(wav_p) / TARGET_SR)
        wav_p, wav_g = align_wav(wav_p, wav_g)
        mel_p = compute_log_mel_norm(wav_p)
        mel_g = compute_log_mel_norm(wav_g)
        rows.append({
            'cond': label,
            'cond_label': pretty_cond(label),
            'sound_id': sound_id,
            'sample_id': sample_id,
            'key_id': f'{sound_id}_{sample_id}',
            'sound_class': sound_meta.loc[sound_meta['sound_id'] == sound_id, 'sound_class'].iloc[0] if (sound_meta['sound_id'] == sound_id).any() else 'unknown',
            'ssim': safe_ssim(mel_p, mel_g),
            'spec_corr': safe_corr(mel_p, mel_g),
            'clap_similarity': cosine_sim(emb_pred[i], emb_gt[i]),
            'pred_path': pred_paths[i],
            'gt_path': gt_paths[i],
        })
    return pd.DataFrame(rows)

results = []
full_df = evaluate_condition(full_index, 'full_model')
if not full_df.empty:
    results.append(full_df)
for cond in ONLY_DIRS:
    df = evaluate_condition(abl_indexes[cond], cond)
    if not df.empty:
        results.append(df)

metrics_df = pd.concat(results, ignore_index=True)
display(metrics_df.head())
print('sample rows:', len(metrics_df))


In [ ]:
global_summary = (
    metrics_df
    .groupby(['cond', 'cond_label'])
    .agg(
        n_samples=('key_id', 'count'),
        ssim_mean=('ssim', 'mean'),
        spec_corr_mean=('spec_corr', 'mean'),
        clap_similarity_mean=('clap_similarity', 'mean'),
    )
    .reset_index()
)
global_summary['sort_key'] = global_summary['cond'].map({c: i for i, c in enumerate(['full_model'] + ONLY_DIRS)})
global_summary = global_summary.sort_values(['sort_key', 'cond_label']).drop(columns='sort_key')
display(global_summary.round(4))

paired_rows = []
full_pairs = metrics_df.loc[metrics_df['cond'] == 'full_model', ['key_id', 'ssim', 'spec_corr', 'clap_similarity']]
for cond in ONLY_DIRS:
    other = metrics_df.loc[metrics_df['cond'] == cond, ['key_id', 'ssim', 'spec_corr', 'clap_similarity']]
    merged = full_pairs.merge(other, on='key_id', suffixes=('_full', '_abl'))
    for metric in ['ssim', 'spec_corr', 'clap_similarity']:
        a = merged[f'{metric}_full'].dropna()
        b = merged[f'{metric}_abl'].dropna()
        if len(a) >= 5 and len(a) == len(b):
            stat, p_value = wilcoxon(a, b)
            delta = float((b - a).mean())
            n_pairs = len(a)
        else:
            stat, p_value, delta, n_pairs = np.nan, np.nan, np.nan, len(merged)
        paired_rows.append({
            'cond': cond,
            'cond_label': pretty_cond(cond),
            'metric': metric,
            'n_pairs': n_pairs,
            'delta_ablation_minus_full': delta,
            'wilcoxon_stat': stat,
            'p_value': p_value,
        })
paired_stats_df = pd.DataFrame(paired_rows)
# display(paired_stats_df.round(5))

# fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
# axes = axes.ravel()
# order = [pretty_cond(c) for c in ['full_model'] + ONLY_DIRS]
# for ax, metric in zip(axes, ['ssim', 'spec_corr', 'clap_similarity']):
#     sns.boxplot(data=metrics_df, x='cond_label', y=metric, order=order, color='#d9d9d9', fliersize=1, ax=ax)
#     sns.stripplot(data=metrics_df.sample(min(len(metrics_df), 1500), random_state=0), x='cond_label', y=metric,
#                   order=order, color='black', alpha=0.18, size=2, ax=ax)
#     ax.set_title(metric)
#     ax.tick_params(axis='x', rotation=45)
#     ax.set_xlabel('')
# fig.suptitle('Metriche globali sample-level per condition', y=1.02)
# plt.tight_layout()
# plt.show()

plt.figure(figsize=(9, max(4, 0.55 * len(global_summary))))
metrics_for_heat = [('ssim_mean', 'SSIM'), ('spec_corr_mean', 'Spectrogram corr'), ('clap_similarity_mean', 'CLAP similarity')]
fig, axes = plt.subplots(1, 3, figsize=(14, max(4, 0.55 * len(global_summary))), sharey=True)
for ax, (col, title) in zip(axes, metrics_for_heat):
    heat = global_summary.set_index('cond_label')[[col]]
    sns.heatmap(heat, annot=True, fmt='.3f', cmap='magma', ax=ax, cbar=True)
    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel('condition')
plt.suptitle('Global average values ​​by condition (separate scale by metric)', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
class_summary = (
    metrics_df
    .groupby(['sound_class', 'cond', 'cond_label'])
    .agg(
        n_samples=('key_id', 'count'),
        spec_corr=('spec_corr', 'mean'),
        clap_similarity=('clap_similarity', 'mean'),
    )
    .reset_index()
)
# display(class_summary.round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, metric in zip(axes, ['spec_corr', 'clap_similarity']):
    pivot = (
        class_summary
        .pivot_table(index='cond_label', columns='sound_class', values=metric)
        .reindex(index=[pretty_cond(c) for c in ['full_model'] + ONLY_DIRS])
        .reindex(columns=CLASS_ORDER)
    )
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='magma', ax=ax)
    ax.set_title(f'{metric} per class')
    ax.set_xlabel('class')
    ax.set_ylabel('condition')
plt.tight_layout()
plt.show()


# Identification Accuracy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

CONDITIONS_TO_EVAL = {
    'ip_adapter':    DIRS['ip_adapter'],
    'brain_only':    DIRS['brain_only'],
    'empty_plus':    DIRS['empty_plus'],
    'brain_to_text': DIRS['brain_to_text'],
}

def pairwise_identification_accuracy_clap(pred_paths, gt_paths, sound_idxs, batch_size=8):
    """
    Calculate pairwise identification accuracy à la Kamitani et al.

    For each sample i:
    - sim_true = pearsonr(emb_pred[i], emb_gt[i])
    - sim_lure = pearsonr(emb_pred[i], emb_gt[j]) for each j != i
    - correct_i = mean(sim_true > sim_lure_j for each j)
    accuracy = mean(correct_i over all samples)

    Returns: scalar accuracy, array of accuracies per sample
    """
    print(f'  → CLAP embeddings pred ({len(pred_paths)} samples)...')
    emb_pred = get_clap_embeddings(pred_paths, batch_size=batch_size)  # (N, D)
    print(f'  → CLAP embeddings GT ...')
    emb_gt   = get_clap_embeddings(gt_paths,   batch_size=batch_size)  # (N, D)

    N = len(emb_pred)
    sim_matrix = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            sim_matrix[i, j] = float(np.corrcoef(emb_pred[i], emb_gt[j])[0, 1])

    # Pairwise identification: for each i, compare sim(pred_i, gt_i) vs sim(pred_i, gt_j) j≠i
    per_sample_acc = []
    for i in range(N):
        sim_true  = sim_matrix[i, i]
        sim_lures = np.concatenate([sim_matrix[i, :i], sim_matrix[i, i+1:]])  # (N-1,)
        n_correct = np.sum(sim_true > sim_lures)
        per_sample_acc.append(n_correct / (N - 1))

    accuracy = float(np.mean(per_sample_acc))
    return accuracy, np.array(per_sample_acc)



In [ ]:
id_acc_rows = []

for label, pred_dir in CONDITIONS_TO_EVAL.items():
    print(f'\n→ {label}')
    matched = [(i, p) for i, p in build_pred_list(pred_dir) if i in gt_index]
    if not matched:
        print(f'  [WARN] no matches found in {pred_dir}')
        continue
    print(f'  {len(matched)} matches')

    pred_paths = [p for _, p in matched]
    gt_paths   = [gt_index[i] for i, _ in matched]
    sound_idxs = [i for i, _ in matched]

    acc, per_sample = pairwise_identification_accuracy_clap(pred_paths, gt_paths, sound_idxs)
    print(f'  Identification accuracy: {acc:.4f} ({acc*100:.1f}%)')

    id_acc_rows.append({
        'condition':        label,
        'id_accuracy_clap': acc,
        'n_pairs':          len(matched),
        'n_lures':          len(matched) - 1,
    })

id_acc_df = pd.DataFrame(id_acc_rows)
ORDER = ['ip_adapter', 'brain_only', 'empty_plus', 'brain_to_text']
id_acc_df['_sort'] = id_acc_df['condition'].map({c: i for i, c in enumerate(ORDER)})
id_acc_df = id_acc_df.sort_values('_sort').drop(columns='_sort')

print('\n=== Identification Accuracy CLAP (pairwise, Pearson, Kamitani-style) ===')
display(id_acc_df.round(4))


# ── Visualizzazione ──────────────────────────────────────────────────────────
COLORS_COND = {
    'ip_adapter':    '#1D9E75',
    'brain_only':    '#D85A30',
    'empty_plus':    '#378ADD',
    'brain_to_text': '#9B59B6',
}

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    id_acc_df['condition'],
    id_acc_df['id_accuracy_clap'] * 100,
    color=[COLORS_COND.get(c, '#888888') for c in id_acc_df['condition']],
    edgecolor='white', linewidth=0.8, width=0.5
)
ax.bar_label(bars, fmt='{:.1f}%', padding=3, fontsize=10)

n_stimuli = id_acc_df['n_pairs'].iloc[0]
chance = 100 / n_stimuli  # ~2% con 50 stimuli (1/(N-1) ≈ 1/49)
ax.axhline(50, color='gray', linestyle='--', linewidth=1, label='50% (intuitive threshold)')
ax.axhline(chance, color='red', linestyle=':', linewidth=1,
           label=f'Chance (1/{n_stimuli-1} ≈ {chance:.1f}%)')
ax.set_ylim(0, 105)
ax.set_ylabel('Identification Accuracy (%)')
ax.set_xlabel('Condition')
ax.set_title('CLAP Identification Accuracy — test set\n(pairwise Pearson, as Kamitani et al.)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# See Spectrograms and Audio

In [ ]:

plot_specs = {'ground_truth': {}}
for cond in metrics_df['cond'].unique():
    plot_specs[cond] = {}

for row in metrics_df.itertuples():
    key = (int(row.sound_id), int(row.sample_id))
    if key not in plot_specs['ground_truth']:
        wav_g = load_audio_48k(row.gt_path)
        plot_specs['ground_truth'][key] = compute_log_mel_norm(wav_g)
    if key not in plot_specs[row.cond]:
        wav_p = load_audio_48k(row.pred_path)
        plot_specs[row.cond][key] = compute_log_mel_norm(wav_p)

def mean_spec_from_keys(cond, keys):
    mats = [plot_specs[cond][key] for key in keys if key in plot_specs.get(cond, {})]
    if not mats:
        return None
    t = min(mat.shape[1] for mat in mats)
    return np.stack([mat[:, :t] for mat in mats], axis=0).mean(axis=0)


In [ ]:
present_classes = [cls for cls in CLASS_ORDER if (metrics_df['sound_class'] == cls).any()]
plot_conds = ['ground_truth', 'full_model'] + ONLY_DIRS

all_means = []
for cls in present_classes:
    cls_rows = metrics_df[metrics_df['sound_class'] == cls][['sound_id', 'sample_id']].drop_duplicates()
    cls_keys = [(int(r.sound_id), int(r.sample_id)) for r in cls_rows.itertuples()]
    for cond in plot_conds:
        ms = mean_spec_from_keys(cond, cls_keys)
        if ms is not None:
            all_means.append(ms)

vmin = min(m.min() for m in all_means)
vmax = max(m.max() for m in all_means)

fig, axes = plt.subplots(
    len(present_classes), len(plot_conds),
    figsize=(2.8 * len(plot_conds), 2.5 * len(present_classes)),
    sharex=True, sharey=True
)
axes = np.atleast_2d(axes)

for r, cls in enumerate(present_classes):
    cls_rows = metrics_df[metrics_df['sound_class'] == cls][['sound_id', 'sample_id']].drop_duplicates()
    cls_keys = [(int(x.sound_id), int(x.sample_id)) for x in cls_rows.itertuples()]
    for c, cond in enumerate(plot_conds):
        ax = axes[r, c]
        ms = mean_spec_from_keys(cond, cls_keys)
        if ms is None:
            ax.axis('off')
            continue
        im = ax.imshow(
            ms, aspect='auto', origin='lower', cmap='magma', vmin=vmin, vmax=vmax,
            extent=[0, 1.0, MEL_FREQS[0] / 1000.0, MEL_FREQS[-1] / 1000.0]
        )
        if r == 0:
            ax.set_title('GT' if cond == 'ground_truth' else pretty_cond(cond), fontsize=9)
        if c == 0:
            ax.set_ylabel(cls)
        ax.set_xticks([])
        ax.set_yticks([])

fig.colorbar(im, ax=axes[:, -1], shrink=0.6, label='normalized log-mel')
fig.suptitle('Spettrogrammi medi per classe e condizione', y=1.02)
plt.tight_layout()
plt.show()


# Gammatone Filterbank

In [ ]:
import numpy as np
import librosa
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tqdm import tqdm
import scipy.io
from scipy.signal import lfilter

# ── Load COCHBA only once ────────────────────────────────────────────────────
mat    = scipy.io.loadmat('.../data/nsltools/aud24.mat')
COCHBA = mat['COCHBA']   # (26, 129) complex128
L, M   = COCHBA.shape    # L=26, M=129

# wav2aud parameters (Santoro 2017: wav2aud(x, [8, 8, 0.1, 0]))
FRMLEN      = 8
TC          = 8
FAC         = 0.1
SHFT        = 0
SR_NSL      = 16000
HAIRCELL_TC = 0.5

# aud2cor parameters (Santoro 2017)
FRAME_RATE = 125
SPEC_MODS  = [0.5, 1, 2, 4]
TEMP_MODS  = [1, 3, 6.25, 12.5, 25]

ONLY_DIRS = [
    'only_HG2hem', 'only_PP2hem', 'only_PT2hem', 'only_aSTG2hem',
    'only_mSTG2hem', 'only_pSTG2hem',
]

# ── Real NSL center frequencies from wav2aud.m documentation ────────────────
# CF = 440 * 2^((-31:97)/24), 128 channels, low→high
CF_NSL  = 440.0 * 2**(np.arange(-31, 97) / 24.0)   # (128,) Hz, low→high
CF_KHZ  = CF_NSL / 1000.0

# ── 1. sigmoid.m ─────────────────────────────────────────────────────────────
def sigmoid_nsl(x, fac):
    if fac > 0:
        return np.log1p(fac * np.maximum(x, 0)) / np.log1p(fac)
    elif fac == 0:
        return (x > 0).astype(float)
    elif fac == -1:
        return np.maximum(x, 0)
    else:
        return x

# ── 2. wav2aud.m ─────────────────────────────────────────────────────────────
def compute_cochleagram(wav, sr, fac=FAC, tc=TC, frmlen=FRMLEN, shft=SHFT):
    """
    Exact replica of wav2aud.m (NSL Tools).
    Output: (128, N_frames) low→high.
    """
    if wav.ndim > 1:
        wav = wav.mean(-1)
    wav = wav.astype(np.float64)
    if sr != SR_NSL:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SR_NSL)

    L_x   = len(wav)
    L_frm = int(round(frmlen * 2**(4 + shft)))
    N     = int(np.ceil(L_x / L_frm))
    wav   = np.append(wav, np.zeros(N * L_frm - L_x))

    alph = np.exp(-1.0 / (tc * 2**(4 + shft))) if tc > 0 else 0.0
    beta = np.exp(-1.0 / (HAIRCELL_TC * 2**(4 + shft)))

    v5 = np.zeros((N, M - 1))

    # Highest channel, lateral inhibition initialization
    ch = M - 1
    p  = int(np.real(COCHBA[0, ch]))
    B  = np.real(COCHBA[1:p+2, ch])
    A  = np.imag(COCHBA[1:p+2, ch])
    y1   = lfilter(B, A, wav)
    y2   = sigmoid_nsl(y1, fac)
    y2   = lfilter([1.0], [1.0, -beta], y2)
    y2_h = y2.copy()

    # High→low loop
    for ch in range(M - 2, -1, -1):
        p = int(np.real(COCHBA[0, ch]))
        B = np.real(COCHBA[1:p+2, ch])
        A = np.imag(COCHBA[1:p+2, ch])

        y1 = lfilter(B, A, wav)
        y2 = sigmoid_nsl(y1, fac)
        y2 = lfilter([1.0], [1.0, -beta], y2)

        y3   = y2 - y2_h
        y2_h = y2.copy()

        y4 = np.maximum(y3, 0)

        if alph:
            y5 = lfilter([1.0], [1.0, -alph], y4)
            v5[:, ch] = y5[L_frm - 1 : N * L_frm : L_frm]
        else:
            v5[:, ch] = y4[:N * L_frm].reshape(N, L_frm).mean(axis=1)

    # v5: (N_frames, 128) — column ch=0 = lowest frequency
    # v5.T: (128, N_frames) — row 0 = lowest frequency = low→high
    return v5.T

# ── 3. gen_cort.m ────────────────────────────────────────────────────────────
def gen_cort(fc, L, STF, PASS=None):
    if PASS is None:
        PASS = [2, 3]
    t  = np.arange(L) / STF * fc
    h  = np.sin(2 * np.pi * t) * t**2 * np.exp(-3.5 * t) * fc
    h -= h.mean()
    H0   = np.fft.fft(h, 2 * L)
    A    = np.angle(H0[:L])
    H    = np.abs(H0[:L])
    maxi = np.argmax(H)
    H   /= H[maxi]
    if PASS[0] == 1:
        H[:maxi] = 1.0
    elif PASS[0] == PASS[1]:
        H[maxi+1:] = 1.0
    return H * np.exp(1j * A)

# ── 4. gen_corf.m ────────────────────────────────────────────────────────────
def gen_corf(fc, L, SRF, PASS=None):
    if PASS is None:
        PASS = [2, 3]
    R1 = np.arange(L) / L * SRF / 2 / abs(fc)
    R1 = R1 ** 2
    H  = R1 * np.exp(1 - R1)
    if PASS[0] == 1:
        maxi = np.argmax(H)
        sumH = H.sum()
        H[:maxi] = 1.0
        H = H / H.sum() * sumH
    elif PASS[0] == PASS[1]:
        maxi = np.argmax(H)
        sumH = H.sum()
        H[maxi+1:] = 1.0
        H = H / H.sum() * sumH
    return H  # real-valued

# ── 5. aud2cor.m + magnitude-only back-projection ───────────────────────────
def cortical_backprojection(cochgram):
    """
    cochgram: (128, N_frames) low→high.
    Output:   (128, N_frames) low→high.
    """
    y    = cochgram.T   # (N_frames, 128) low→high
    N, M = y.shape

    rv  = np.array(TEMP_MODS)
    sv  = np.array(SPEC_MODS)
    K1  = len(rv)
    K2  = len(sv)
    STF = FRAME_RATE
    SRF = 24
    BP  = 0

    N1 = int(2**np.ceil(np.log2(N)))
    N2 = N1 * 2
    M1 = int(2**np.ceil(np.log2(M)))
    M2 = M1 * 2

    # 2D FFT, as in aud2cor
    Y = np.zeros((N2, M1), dtype=complex)
    for n in range(N):
        R1 = np.fft.fft(y[n, :], M2)
        Y[n, :] = R1[:M1]
    for m in range(M1):
        R1 = np.fft.fft(Y[:N, m], N2)
        Y[:, m] = R1

    bp_full = np.zeros((N, M))

    for rdx, fc_rt in enumerate(rv):
        HR = gen_cort(fc_rt, N1, STF, [rdx + 1 + BP, K1 + BP * 2])

        for sgn in [1, -1]:
            if sgn > 0:
                HR_dsb = np.concatenate([HR, np.zeros(N1)])
            else:
                HR_dsb = np.concatenate([
                    HR[:1],
                    np.conj(HR[1:][::-1]),
                    np.zeros(N2 - N1)
                ])
                HR_dsb[N1] = np.abs(HR[min(N1 + 1, len(HR) - 1)])

            z1 = np.zeros((N2, M1), dtype=complex)
            for m in range(M1):
                z1[:, m] = HR_dsb * Y[:, m]
            z1 = np.fft.ifft(z1, axis=0)
            z1 = z1[:N, :]

            for sdx, fc_sc in enumerate(sv):
                HS = gen_corf(fc_sc, M1, SRF, [sdx + 1 + BP, K2 + BP * 2])
                z  = np.zeros((N, M), dtype=complex)
                for n in range(N):
                    R1 = np.fft.ifft(z1[n, :] * HS, M2)
                    z[n, :] = R1[:M]
                bp_full += np.abs(z)

    bp_full /= (K1 * K2 * 2)
    return bp_full.T   # (128, N_frames) low→high

# ── 6. Pipeline ──────────────────────────────────────────────────────────────
def compute_cortical_repr(wav, sr):
    coch = compute_cochleagram(wav, sr)  # (128, N) low→high
    bp   = cortical_backprojection(coch) # (128, N) low→high
    return bp

# ── 7. Cache ─────────────────────────────────────────────────────────────────
plot_specs = {'ground_truth': {}}
for cond in metrics_df['cond'].unique():
    plot_specs[cond] = {}

for row in tqdm(list(metrics_df.itertuples()), desc='Cortical representations'):
    key = (int(row.sound_id), int(row.sample_id))
    if key not in plot_specs['ground_truth']:
        wav_g, sr_g = librosa.load(row.gt_path, sr=None, mono=True)
        plot_specs['ground_truth'][key] = compute_cortical_repr(wav_g, sr_g)
    if key not in plot_specs[row.cond]:
        wav_p, sr_p = librosa.load(row.pred_path, sr=None, mono=True)
        plot_specs[row.cond][key] = compute_cortical_repr(wav_p, sr_p)

# ── 8. Average by class ──────────────────────────────────────────────────────
def mean_spec_from_keys(cond, keys):
    mats = [plot_specs[cond][key] for key in keys if key in plot_specs.get(cond, {})]
    if not mats:
        return None
    t = min(m.shape[1] for m in mats)
    return np.stack([m[:, :t] for m in mats], axis=0).mean(axis=0)

present_classes = [cls for cls in CLASS_ORDER if (metrics_df['sound_class'] == cls).any()]
plot_conds      = ['ground_truth', 'full_model'] + ONLY_DIRS

class_means = {}
for cls in tqdm(present_classes, desc='Averaging by class'):
    cls_rows = metrics_df[metrics_df['sound_class'] == cls][['sound_id','sample_id']].drop_duplicates()
    cls_keys = [(int(r.sound_id), int(r.sample_id)) for r in cls_rows.itertuples()]
    class_means[cls] = {}
    for cond in plot_conds:
        class_means[cls][cond] = mean_spec_from_keys(cond, cls_keys)

# ── 9. y-axis ────────────────────────────────────────────────────────────────
ytick_khz = [0.25, 0.5, 1, 2, 4]
ytick_idx = [np.argmin(np.abs(CF_KHZ - f)) for f in ytick_khz]

# ── 10. Plot ─────────────────────────────────────────────────────────────────
n_rows = len(present_classes)
n_cols = len(plot_conds)

fig = plt.figure(figsize=(2.6 * n_cols + 1.2, 2.6 * n_rows))
gs  = gridspec.GridSpec(
    n_rows, n_cols + 1,
    figure=fig,
    width_ratios=[1.0] * n_cols + [0.04],
    hspace=0.06,
    wspace=0.04,
)
axes    = np.array([[fig.add_subplot(gs[r, c]) for c in range(n_cols)] for r in range(n_rows)])
cbar_ax = fig.add_subplot(gs[:, -1])

im_last = None
for r, cls in enumerate(tqdm(present_classes, desc='Plotting')):
    gt_ms    = class_means[cls]['ground_truth']
    vmin_cls = np.percentile(gt_ms, 2)
    vmax_cls = np.percentile(gt_ms, 98)

    for c, cond in enumerate(plot_conds):
        ax = axes[r, c]
        ms = class_means[cls][cond]
        if ms is None:
            ax.axis('off')
            continue

        im_last = ax.imshow(
            ms,
            aspect='auto',
            origin='lower',
            cmap='jet',
            vmin=vmin_cls,
            vmax=vmax_cls,
        )

        if r == 0:
            label = 'GT' if cond == 'ground_truth' else pretty_cond(cond)
            ax.set_title(label, fontsize=8, pad=3)

        if c == 0:
            ax.set_ylabel(cls, fontsize=8)
            ax.set_yticks(ytick_idx)
            ax.set_yticklabels([f'{f}' for f in ytick_khz], fontsize=6)
        else:
            ax.set_yticks([])

        ax.set_xticks([])

if im_last is not None:
    cb = fig.colorbar(im_last, cax=cbar_ax)
    cb.set_label('cortical repr. — GT scale per class', fontsize=7)
    cb.ax.tick_params(labelsize=6)

fig.suptitle(
    'Average cortical representation by class and condition\n'
    '(NSL wav2aud [8,8,0.1,0] + aud2cor back-projection)',
    fontsize=10, y=1.01,
)
plt.tight_layout()
plt.show()

# Single trial visualization

In [ ]:
available_pairs = (
    metrics_df[['sound_id', 'sample_id', 'sound_class', 'gt_path']]
    .drop_duplicates()
    .sort_values(['sample_id', 'sound_id'])
    .reset_index(drop=True)
)
display(available_pairs.head(20))


In [ ]:
# Example: `..._sound0084_sample0021.wav` -> SELECT_SAMPLE_ID = 21
SELECT_SAMPLE_ID = 33
SELECT_SOUND_ID = None
SHOW_AUDIO_PLAYERS = True

sample_selector = metrics_df[metrics_df['sample_id'] == SELECT_SAMPLE_ID]
if SELECT_SOUND_ID is not None:
    sample_selector = sample_selector[sample_selector['sound_id'] == SELECT_SOUND_ID]

if sample_selector.empty:
    raise ValueError(f'No sample found for sample_id={SELECT_SAMPLE_ID} and sound_id={SELECT_SOUND_ID}')

sel = sample_selector[['sound_id', 'sample_id', 'sound_class', 'gt_path']].drop_duplicates().iloc[0]
selected_key = (int(sel['sound_id']), int(sel['sample_id']))
selected_class = sel['sound_class']
selected_gt_path = sel['gt_path']

print(f'Selected key: sound={selected_key[0]}, sample={selected_key[1]}, class={selected_class}')

cond_to_path = {'ground_truth': selected_gt_path}
full_hit = metrics_df[(metrics_df['sound_id'] == selected_key[0]) & (metrics_df['sample_id'] == selected_key[1]) & (metrics_df['cond'] == 'full_model')]
if not full_hit.empty:
    cond_to_path['full_model'] = full_hit.iloc[0]['pred_path']
for cond in ONLY_DIRS:
    hit = metrics_df[(metrics_df['sound_id'] == selected_key[0]) & (metrics_df['sample_id'] == selected_key[1]) & (metrics_df['cond'] == cond)]
    if not hit.empty:
        cond_to_path[cond] = hit.iloc[0]['pred_path']

panel_conds = [cond for cond in ['ground_truth', 'full_model'] + ONLY_DIRS if cond in cond_to_path]
panel_specs = {}
for cond in panel_conds:
    wav = load_audio_48k(cond_to_path[cond])
    panel_specs[cond] = compute_log_mel_norm(wav)

min_t = min(spec.shape[1] for spec in panel_specs.values())
panel_specs = {cond: spec[:, :min_t] for cond, spec in panel_specs.items()}
vmin = min(spec.min() for spec in panel_specs.values())
vmax = max(spec.max() for spec in panel_specs.values())

fig, axes = plt.subplots(2, len(panel_conds), figsize=(3.2 * len(panel_conds), 6), sharex=True, sharey='row')
axes = np.atleast_2d(axes)
gt_spec = panel_specs['ground_truth']

for i, cond in enumerate(panel_conds):
    spec = panel_specs[cond]
    axes[0, i].imshow(spec, aspect='auto', origin='lower', cmap='magma', vmin=vmin, vmax=vmax,
                      extent=[0, 1.0, MEL_FREQS[0] / 1000.0, MEL_FREQS[-1] / 1000.0])
    axes[0, i].set_title('GT' if cond == 'ground_truth' else pretty_cond(cond), fontsize=9)
    axes[0, i].set_xticks([])
    axes[0, i].set_yticks([])

    diff = spec - gt_spec
    axes[1, i].imshow(diff, aspect='auto', origin='lower', cmap='coolwarm', vmin=-0.6, vmax=0.6,
                      extent=[0, 1.0, MEL_FREQS[0] / 1000.0, MEL_FREQS[-1] / 1000.0])
    axes[1, i].set_xticks([])
    axes[1, i].set_yticks([])

axes[0, 0].set_ylabel('log-mel')
axes[1, 0].set_ylabel('diff vs GT')
fig.suptitle(f'Single sample: sound={selected_key[0]}, sample={selected_key[1]}, class={selected_class}', y=1.02)
plt.tight_layout()
plt.show()

if SHOW_AUDIO_PLAYERS:
    print('Audio players:')
    for cond in panel_conds:
        label = 'GT' if cond == 'ground_truth' else pretty_cond(cond)
        print(label)
        display(Audio(cond_to_path[cond]))